# Supervised AIEDRL reproduction

This notebook is the supported entry point for reproducing the submitted experiment. It deliberately processes one problem at a time by default because generated code or external model calls may require operator attention. The historical exploratory notebook remains in `agents_and_training.ipynb`.

Before running: create `secrets/openai.env` from the example, review `configs/paper_reproduction.toml`, and use a new `run_id` for a logically separate run.

In [ ]:
from pathlib import Path

from experiment import create_runner_from_config

CONFIG_PATH = Path("configs/paper_reproduction.toml")
runner = create_runner_from_config(CONFIG_PATH)
print(f"Run directory: {runner.config.run_path}")
print(f"Configured problems: {runner.config.run.problem_start}..{runner.config.run.problem_stop - 1}")

## Inspect status

Run this cell after a restart. An attempt that began but never wrote a terminal event is automatically marked `interrupted`. Failed and interrupted problems are not silently retried.

In [ ]:
status = runner.status()
display(status.groupby("status").size().rename("problems"))
display(status[status["status"].isin(["failed", "interrupted"])].head(20))

## Run the next problem

Set the confirmation flag only when you intend to make live API calls. Increase `batch_size` cautiously; every completed turn and attempt is checkpointed.

In [ ]:
CONFIRM_LIVE_API_RUN = False

if CONFIRM_LIVE_API_RUN:
    results = runner.run_next(batch_size=1)
    display(results)
else:
    print("No API calls made. Set CONFIRM_LIVE_API_RUN = True to run the next problem.")

## Retry or skip an attended problem

Use `retry` only for a problem whose status is `failed` or `interrupted`. Use `skip` when the operator decides not to retry; a non-empty reason is required.

In [ ]:
# result = runner.retry(problem_id=4)
# runner.skip(problem_id=4, reason="Repeated provider failure after two supervised retries")
# display(runner.status().query("problem_id == 4"))